# Crescimento, Interpolação e Extrapolação Populacional — Método Geométrico
### Todos os Municípios de MG (1970-2023)

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np

from utilitarios_populacional import (
    COLUNA_MUNICIPIO,
    COLUNA_CODIGO_IBGE,
    POPULACAO_MINIMA_VALIDA,
    calcular_taxa_geometrica,
    estimar_populacao_geometrica,
    calcular_taxa_exponencial,
    estimar_populacao_exponencial,
)

In [2]:
CAMINHO_ARQUIVO_ENTRADA = Path('/home/anna/tcc/Seleção e Transformação/População - Faixa Etária/tabela_populacao_faixa_etaria.csv')
CAMINHO_ARQUIVO_SAIDA = Path('/home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/populacao_interpolacao_extrapolacao_geometrica.csv')

LISTA_ANOS_CENSITARIOS = [1970, 1980, 1991, 2000, 2010, 2022]
ANO_INICIO = 1970
ANO_FIM = 2025 
LISTA_ANOS_COMPLETOS = list(range(ANO_INICIO, ANO_FIM + 1))

## Leitura e Validação dos Anos Censitários

Confere se todas as colunas `TOTAL_<ano censitário>` esperadas realmente existem no arquivo de entrada, e se correspondem exatamente aos anos censitários definidos em `LISTA_ANOS_CENSITARIOS`.

In [ ]:
tabela_populacao = pd.read_csv(CAMINHO_ARQUIVO_ENTRADA)

lista_colunas_totais = [f"TOTAL_{ano}" for ano in LISTA_ANOS_CENSITARIOS]

colunas_faltantes = [coluna for coluna in lista_colunas_totais if coluna not in tabela_populacao.columns]
if colunas_faltantes:
    raise ValueError(
        f"As seguintes colunas de anos censitários não foram encontradas no arquivo de entrada: {colunas_faltantes}. "
        f"Colunas TOTAL_* disponíveis: {[c for c in tabela_populacao.columns if c.startswith('TOTAL_')]}"
    )

anos_encontrados = []

for coluna in tabela_populacao.columns:
    if coluna.startswith("TOTAL_"):
        ano_texto = coluna.replace("TOTAL_", "")
        ano_inteiro = int(ano_texto)
        anos_encontrados.append(ano_inteiro)

# Ordena a lista de anos do menor para o maior
anos_disponiveis_no_arquivo = sorted(anos_encontrados)
print("Anos censitários esperados  :", LISTA_ANOS_CENSITARIOS)
print("Anos censitários no arquivo :", anos_disponiveis_no_arquivo)

assert LISTA_ANOS_CENSITARIOS == anos_disponiveis_no_arquivo, (
    "Os anos censitários definidos em LISTA_ANOS_CENSITARIOS não coincidem "
    "exatamente com os anos TOTAL_* disponíveis no arquivo de entrada. "
    "Revise a configuração antes de prosseguir."
)

print("\nTodos os anos censitários conferem com o arquivo de entrada.")

tabela_totais = (
    tabela_populacao[[COLUNA_MUNICIPIO, COLUNA_CODIGO_IBGE] + lista_colunas_totais]
    .copy()
    .set_index(COLUNA_CODIGO_IBGE, drop=False)
)
tabela_totais.head()

Anos censitários esperados  : [1970, 1980, 1991, 2000, 2010, 2022]
Anos censitários no arquivo : [1970, 1980, 1991, 2000, 2010, 2022]

Correto — todos os anos censitários conferem com o arquivo de entrada.


,MUNICÍPIOS,CÓDIGO IBGE,TOTAL_1970,TOTAL_1980,TOTAL_1991,TOTAL_2000,TOTAL_2010,TOTAL_2022
CÓDIGO IBGE,,,,,,,,
3100104,abadia dos dourados,3100104,4848,3936,3252,3105,3676,3111
3100203,abaete,3100203,10206,9926,10560,10947,11171,11474
3100302,abre campo,3100302,9779,9375,9131,6610,7238,6903
3100401,acaiaca,3100401,1699,1822,1937,1875,1881,2012
3100807,aguanil,3100807,1312,1560,1576,1753,1938,2196


## Checagem de Valores Não Positivos


In [4]:
municipios_com_valor_nao_positivo = tabela_totais[
    (tabela_totais[lista_colunas_totais] <= 0).any(axis=1)
]

if len(municipios_com_valor_nao_positivo) > 0:
    print(f"ATENÇÃO: {len(municipios_com_valor_nao_positivo)} município(s) com população <= 0 em algum ano censitário:")
    display(municipios_com_valor_nao_positivo[[COLUNA_MUNICIPIO, COLUNA_CODIGO_IBGE] + lista_colunas_totais])
else:
    print("Nenhum município com população <= 0 em nenhum ano censitário. Ok para prosseguir com log().")

ATENÇÃO: 131 município(s) com população <= 0 em algum ano censitário:


,MUNICÍPIOS,CÓDIGO IBGE,TOTAL_1970,TOTAL_1980,TOTAL_1991,TOTAL_2000,TOTAL_2010,TOTAL_2022
CÓDIGO IBGE,,,,,,,,
3101631,alfredo vasconcelos,3101631,0,0,467,2581,2443,3471
3102050,alto caparao,3102050,0,0,385,2368,2135,2880
3102852,angelandia,3102852,0,0,505,3970,3475,3916
3103751,arapora,3103751,0,0,521,2715,2470,4246
3104452,aricanduva,3104452,0,0,294,2149,2020,2317
...,...,...,...,...,...,...,...,...
3170651,vargem grande do rio pardo,3170651,0,0,365,2362,1959,2248
3170750,varjao de minas,3170750,0,0,443,2420,2398,3354
3171030,verdelandia,3171030,0,0,517,3865,3546,3762


## Interpolação Geométrica 

$$r_g = \left(\dfrac{P^{final}}{P^{inicial}}\right)^{1/t} - 1 \qquad P^t = P^{inicial} \cdot (1+r_g)^t$$


In [6]:
def interpolar_geometrica(tabela, colunas_censitarias, anos_censitarios, anos_completos):

    P = tabela[colunas_censitarias].copy()
    P.columns = anos_censitarios

    resultado = pd.DataFrame(index=P.index, columns=anos_completos, dtype=float)

    for ano in anos_censitarios:
        resultado[ano] = P[ano]

    for i in range(len(anos_censitarios) - 1):
        ano_inicial = anos_censitarios[i]
        ano_final = anos_censitarios[i + 1]

        populacao_inicial = P[ano_inicial]   # P^inicial da Fórmula 1
        populacao_final = P[ano_final]       # P^final da Fórmula 1
        tempo_total = ano_final - ano_inicial  # "t" da Fórmula 1 (intervalo entre os censos)

        # Fórmula 1 -
        r_g = calcular_taxa_geometrica(populacao_inicial, populacao_final, tempo_total)

        # Preenche cada ano dentro deste intervalo com a Fórmula 3
        anos_intermediarios = [ano for ano in anos_completos if ano_inicial < ano < ano_final]
        for ano in anos_intermediarios:
            tempo_decorrido = ano - ano_inicial  # "t" da Fórmula 3 (desde o início do intervalo)
            resultado[ano] = estimar_populacao_geometrica(populacao_inicial, r_g, tempo_decorrido)

    return resultado

tabela_interpolacao_geometrica = interpolar_geometrica(
    tabela_totais, lista_colunas_totais, LISTA_ANOS_CENSITARIOS, LISTA_ANOS_COMPLETOS
)
tabela_interpolacao_geometrica.columns = [f"TOTAL_{ano}" for ano in LISTA_ANOS_COMPLETOS]
tabela_interpolacao_geometrica.head()

,TOTAL_1970,TOTAL_1971,TOTAL_1972,TOTAL_1973,TOTAL_1974,TOTAL_1975,TOTAL_1976,TOTAL_1977,TOTAL_1978,TOTAL_1979,...,TOTAL_2016,TOTAL_2017,TOTAL_2018,TOTAL_2019,TOTAL_2020,TOTAL_2021,TOTAL_2022,TOTAL_2023,TOTAL_2024,TOTAL_2025
CÓDIGO IBGE,,,,,,,,,,,,,,,,,,,,,
3100104,4848,4748.012559,4650.087306,4554.181709,4460.254115,4368.263728,4278.170594,4189.935584,4103.520374,4018.887432,...,3381.720864,3335.017619,3288.959369,3243.537206,3198.742346,3154.566125,3111,NaN,NaN,NaN
3100203,10206,10177.648193,10149.375145,10121.180639,10093.064456,10065.026379,10037.066189,10009.183673,9981.378612,9953.650793,...,11321.486387,11346.763809,11372.097669,11397.488092,11422.935203,11448.439130,11474,NaN,NaN,NaN
3100302,9779,9737.828669,9696.830677,9656.005294,9615.351793,9574.869451,9534.557546,9494.415362,9454.442183,9414.637299,...,7068.515686,7040.656676,7012.907466,6985.267624,6957.736718,6930.314319,6903,NaN,NaN,NaN
3100401,1699,1710.916751,1722.917085,1735.001589,1747.170854,1759.425474,1771.766048,1784.193178,1796.707472,1809.309541,...,1945.397646,1956.342926,1967.349788,1978.418576,1989.549641,2000.743331,2012,NaN,NaN,NaN
3100807,1312,1334.912843,1358.225837,1381.945971,1406.080354,1430.636222,1455.620935,1481.041982,1506.906984,1533.223694,...,2062.970674,2084.568966,2106.393381,2128.446288,2150.730077,2173.247167,2196,NaN,NaN,NaN


## Extrapolação Exponencial


$$P_t = P_0 \, e^{k_g (t - t_0)} \qquad k_g = \dfrac{\ln P_2 - \ln P_0}{t_2 - t_0}$$

onde $P_t$ representa a população estimada no ano $t$; $P_0$ e $P_2$, as populações observadas nos anos $t_0$ e $t_2$, respectivamente (últimos dois anos censitários de referência); e $k_g$, a taxa geométrica de crescimento populacional no período.

In [ ]:
def extrapolar_geometrica(tabela_interpolada, populacao_inicial_col, populacao_final_col,
                           ano_inicial, ano_final, anos_a_extrapolar):

    P_inicial = tabela_interpolada[populacao_inicial_col]
    P_final = tabela_interpolada[populacao_final_col]
    t = ano_final - ano_inicial

    k_g = calcular_taxa_exponencial(P_inicial, P_final, t)

    extrapolacao = pd.DataFrame(index=tabela_interpolada.index)
    for ano in anos_a_extrapolar:
        extrapolacao[f"TOTAL_{ano}"] = estimar_populacao_exponencial(P_final, k_g, ano - ano_final)

    return extrapolacao, k_g

ULTIMOS_DOIS_CENSOS = LISTA_ANOS_CENSITARIOS[-2:]  # [2010, 2022]
ANOS_A_EXTRAPOLAR = [ano for ano in LISTA_ANOS_COMPLETOS if ano > LISTA_ANOS_CENSITARIOS[-1]]

tabela_extrapolacao_geometrica, taxa_exponencial_extrapolacao = extrapolar_geometrica(
    tabela_interpolacao_geometrica,
    populacao_inicial_col=f"TOTAL_{ULTIMOS_DOIS_CENSOS[0]}",
    populacao_final_col=f"TOTAL_{ULTIMOS_DOIS_CENSOS[1]}",
    ano_inicial=ULTIMOS_DOIS_CENSOS[0],
    ano_final=ULTIMOS_DOIS_CENSOS[1],
    anos_a_extrapolar=ANOS_A_EXTRAPOLAR,
)

print("Últimos dois censos usados na taxa de extrapolação:", ULTIMOS_DOIS_CENSOS)
print("Anos extrapolados                                 :", ANOS_A_EXTRAPOLAR)
tabela_extrapolacao_geometrica.head()

## Interpolação + Extrapolação Geométrica (1970-2025)

Combina os resultados das duas etapas anteriores em uma única série contínua por município, do primeiro censo (1970) até o último ano definido em `ANO_FIM`.

In [8]:
tabela_populacao_completa = tabela_interpolacao_geometrica.copy()

for coluna in tabela_extrapolacao_geometrica.columns:
    tabela_populacao_completa[coluna] = tabela_populacao_completa[coluna].fillna(
        tabela_extrapolacao_geometrica[coluna]
    )

colunas_ordenadas = [f"TOTAL_{ano}" for ano in LISTA_ANOS_COMPLETOS]
tabela_populacao_completa = tabela_populacao_completa[colunas_ordenadas]

tabela_populacao_completa.head()

,TOTAL_1970,TOTAL_1971,TOTAL_1972,TOTAL_1973,TOTAL_1974,TOTAL_1975,TOTAL_1976,TOTAL_1977,TOTAL_1978,TOTAL_1979,...,TOTAL_2016,TOTAL_2017,TOTAL_2018,TOTAL_2019,TOTAL_2020,TOTAL_2021,TOTAL_2022,TOTAL_2023,TOTAL_2024,TOTAL_2025
CÓDIGO IBGE,,,,,,,,,,,,,,,,,,,,,
3100104,4848,4748.012559,4650.087306,4554.181709,4460.254115,4368.263728,4278.170594,4189.935584,4103.520374,4018.887432,...,3381.720864,3335.017619,3288.959369,3243.537206,3198.742346,3154.566125,3111,3068.035545,3025.664450,2983.878521
3100203,10206,10177.648193,10149.375145,10121.180639,10093.064456,10065.026379,10037.066189,10009.183673,9981.378612,9953.650793,...,11321.486387,11346.763809,11372.097669,11397.488092,11422.935203,11448.439130,11474,11499.617939,11525.293076,11551.025537
3100302,9779,9737.828669,9696.830677,9656.005294,9615.351793,9574.869451,9534.557546,9494.415362,9454.442183,9414.637299,...,7068.515686,7040.656676,7012.907466,6985.267624,6957.736718,6930.314319,6903,6875.793334,6848.693897,6821.701267
3100401,1699,1710.916751,1722.917085,1735.001589,1747.170854,1759.425474,1771.766048,1784.193178,1796.707472,1809.309541,...,1945.397646,1956.342926,1967.349788,1978.418576,1989.549641,2000.743331,2012,2023.320002,2034.703692,2046.151430
3100807,1312,1334.912843,1358.225837,1381.945971,1406.080354,1430.636222,1455.620935,1481.041982,1506.906984,1533.223694,...,2062.970674,2084.568966,2106.393381,2128.446288,2150.730077,2173.247167,2196,2218.991044,2242.222793,2265.697767


## Arredondamento

População é uma contagem de pessoas, então não faz sentido manter casas decimais nos valores interpolados/extrapolados. Usa-se `np.ceil` (arredondamento para cima) em vez de arredondamento comum, para nunca subestimar a população de nenhum município em nenhum ano.

In [ ]:
colunas_ano = []
for col in tabela_populacao_completa.columns:
    if col.startswith("TOTAL_"):
        colunas_ano.append(col)

tabela_populacao_completa[colunas_ano] = (
    np.ceil(tabela_populacao_completa[colunas_ano])
    .replace([np.inf, -np.inf], np.nan)
    .astype("Int64")
)
tabela_populacao_completa.head()

,TOTAL_1970,TOTAL_1971,TOTAL_1972,TOTAL_1973,TOTAL_1974,TOTAL_1975,TOTAL_1976,TOTAL_1977,TOTAL_1978,TOTAL_1979,...,TOTAL_2016,TOTAL_2017,TOTAL_2018,TOTAL_2019,TOTAL_2020,TOTAL_2021,TOTAL_2022,TOTAL_2023,TOTAL_2024,TOTAL_2025
CÓDIGO IBGE,,,,,,,,,,,,,,,,,,,,,
3100104,4848,4749,4651,4555,4461,4369,4279,4190,4104,4019,...,3382,3336,3289,3244,3199,3155,3111,3069,3026,2984
3100203,10206,10178,10150,10122,10094,10066,10038,10010,9982,9954,...,11322,11347,11373,11398,11423,11449,11474,11500,11526,11552
3100302,9779,9738,9697,9657,9616,9575,9535,9495,9455,9415,...,7069,7041,7013,6986,6958,6931,6903,6876,6849,6822
3100401,1699,1711,1723,1736,1748,1760,1772,1785,1797,1810,...,1946,1957,1968,1979,1990,2001,2012,2024,2035,2047
3100807,1312,1335,1359,1382,1407,1431,1456,1482,1507,1534,...,2063,2085,2107,2129,2151,2174,2196,2219,2243,2266


## Validação Final — Checagem de Valores Ausentes


In [10]:
colunas_ano = [f"TOTAL_{ano}" for ano in LISTA_ANOS_COMPLETOS]
total_ausentes = tabela_populacao_completa[colunas_ano].isna().sum().sum()

print(f"Total de valores ausentes na série {ANO_INICIO}-{ANO_FIM}: {total_ausentes}")

if total_ausentes > 0:
    linhas_com_ausente = tabela_populacao_completa[colunas_ano].isna().any(axis=1)
    municipios_com_ausente = tabela_totais.loc[linhas_com_ausente, COLUNA_MUNICIPIO].tolist()
    print("Municípios com valores ausentes:", municipios_com_ausente)
else:
    print("Série completa — nenhum valor ausente entre", ANO_INICIO, "e", ANO_FIM, ".")

Total de valores ausentes na série 1970-2025: 2479
Municípios com valores ausentes: ['alfredo vasconcelos', 'alto caparao', 'angelandia', 'arapora', 'aricanduva', 'berizal', 'bonito de minas', 'brasilandia de minas', 'bugre', 'cabeceira grande', 'campo azul', 'cantagalo', 'capitao andrade', 'carneirinho', 'catas altas', 'catuji', 'catuti', 'chapada gaucha', 'confins', 'crisolita', 'cuparaque', 'curral de dentro', 'corrego fundo', 'conego marinho', 'delta', 'divisa alegre', 'divisopolis', 'dom bosco', 'durande', 'entre folhas', 'fervedouro', 'franciscopolis', 'frei lagonegro', 'fruta de leite', 'gameleiras', 'glaucilandia', 'goiabeira', 'goiana', 'guaraciama', 'ibiracatu', 'icarai de minas', 'imbe de minas', 'indaiabira', 'ipaba', 'itau de minas', 'jampruca', 'japonvar', 'jaiba', 'jenipapo de minas', 'josenopolis', 'jose goncalves de minas', 'jose raydan', 'juatuba', 'juvenilia', 'lagoa grande', 'leme do prado', 'limeira do oeste', 'lontra', 'luisburgo', 'luislandia', 'mamonas', 'martin

## Exportação 

In [11]:
tabela_saida = tabela_totais[[COLUNA_MUNICIPIO, COLUNA_CODIGO_IBGE]].merge(
    tabela_populacao_completa, left_index=True, right_index=True
)

CAMINHO_ARQUIVO_SAIDA.parent.mkdir(parents=True, exist_ok=True)
tabela_saida.to_csv(CAMINHO_ARQUIVO_SAIDA, index=False)

print("Arquivo salvo em:", CAMINHO_ARQUIVO_SAIDA)
print("Shape:", tabela_saida.shape)
print(f"Período coberto: {ANO_INICIO}-{ANO_FIM} ({len(LISTA_ANOS_COMPLETOS)} anos)")
tabela_saida.head()

Arquivo salvo em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/populacao_interpolacao_extrapolacao_geometrica.csv
Shape: (853, 58)
Período coberto: 1970-2025 (56 anos)


,MUNICÍPIOS,CÓDIGO IBGE,TOTAL_1970,TOTAL_1971,TOTAL_1972,TOTAL_1973,TOTAL_1974,TOTAL_1975,TOTAL_1976,TOTAL_1977,...,TOTAL_2016,TOTAL_2017,TOTAL_2018,TOTAL_2019,TOTAL_2020,TOTAL_2021,TOTAL_2022,TOTAL_2023,TOTAL_2024,TOTAL_2025
CÓDIGO IBGE,,,,,,,,,,,,,,,,,,,,,
3100104,abadia dos dourados,3100104,4848,4749,4651,4555,4461,4369,4279,4190,...,3382,3336,3289,3244,3199,3155,3111,3069,3026,2984
3100203,abaete,3100203,10206,10178,10150,10122,10094,10066,10038,10010,...,11322,11347,11373,11398,11423,11449,11474,11500,11526,11552
3100302,abre campo,3100302,9779,9738,9697,9657,9616,9575,9535,9495,...,7069,7041,7013,6986,6958,6931,6903,6876,6849,6822
3100401,acaiaca,3100401,1699,1711,1723,1736,1748,1760,1772,1785,...,1946,1957,1968,1979,1990,2001,2012,2024,2035,2047
3100807,aguanil,3100807,1312,1335,1359,1382,1407,1431,1456,1482,...,2063,2085,2107,2129,2151,2174,2196,2219,2243,2266


## Comparação — Valor Original vs. Valor Interpolado (Anos Censitários)

Compara, apenas nos anos de censo (1970, 1980, 1991, 2000, 2010, 2022), o valor original do arquivo de entrada com o valor da série interpolada/arredondada. Nesses anos os dois deveriam coincidir (ou diferir no máximo por causa do `np.ceil`), servindo como checagem de sanidade da interpolação.

In [12]:
lista_comparacoes = []

for ano in LISTA_ANOS_CENSITARIOS:
    coluna_ano = f"TOTAL_{ano}"
    comparacao_ano = pd.DataFrame({
        COLUNA_MUNICIPIO: tabela_totais[COLUNA_MUNICIPIO],
        "ANO": ano,
        "POPULACAO_ORIGINAL": tabela_totais[coluna_ano].values,
        "POPULACAO_INTERPOLADA": tabela_saida[coluna_ano].values,
    })
    lista_comparacoes.append(comparacao_ano)

tabela_comparacao_censos = pd.concat(lista_comparacoes, ignore_index=True)

tabela_comparacao_censos["DIFERENCA"] = (
    tabela_comparacao_censos["POPULACAO_INTERPOLADA"] - tabela_comparacao_censos["POPULACAO_ORIGINAL"]
)

print(f"Total de divergências entre original e interpolado nos anos censitários: {(tabela_comparacao_censos['DIFERENCA'] != 0).sum()}")

tabela_comparacao_censos

Total de divergências entre original e interpolado nos anos censitários: 0


,MUNICÍPIOS,ANO,POPULACAO_ORIGINAL,POPULACAO_INTERPOLADA,DIFERENCA
0,abadia dos dourados,1970,4848,4848,0
1,abaete,1970,10206,10206,0
2,abre campo,1970,9779,9779,0
3,acaiaca,1970,1699,1699,0
4,aguanil,1970,1312,1312,0
...,...,...,...,...,...
5113,wenceslau braz,2022,1176,1176,0
5114,agua boa,2022,6214,6214,0
5115,agua comprida,2022,1038,1038,0
5116,aguas formosas,2022,9181,9181,0
